## Complete Summary

### Part 1: Boundary Downloads

| Function | Purpose |
|----------|---------|
| `get_census_boundaries()` | Download TIGER/Line shapefiles |
| `normalize_state_identifier()` | Convert state names/abbrevs to FIPS |
| `get_available_years()` | List available Census years |
| `discover_boundary_types()` | Find boundary types for a year |

### Part 2: Demographics (CensusAPIClient)

| Function | Purpose |
|----------|---------|
| `get_population()` | Fetch population counts |
| `get_demographics()` | Fetch demographic variables |
| `get_income_data()` | Fetch income-related variables |
| `get_census_data_with_geometry()` | Combined: boundaries + demographics in one GeoDataFrame |

### Part 3: GEOID Utilities

| Function | Purpose |
|----------|---------|
| `construct_geoid()` | Build GEOID from components |
| `parse_geoid()` | Extract components from GEOID |
| `normalize_geoid()` | Add leading zeros |
| `validate_geoid()` | Check GEOID format |

**API Key**: Get a free Census API key at https://api.census.gov/data/key_signup.html
Set as `CENSUS_API_KEY` environment variable or pass `api_key` parameter.

In [ ]:
from siege_utilities.geo.geoid_utils import (
    construct_geoid,
    parse_geoid,
    normalize_geoid,
    validate_geoid,
    GEOID_LENGTHS
)

# GEOID lengths by geography type
print("GEOID lengths by geography:")
for geo, length in GEOID_LENGTHS.items():
    print(f"  {geo}: {length} digits")

print("\n" + "=" * 50)

# Construct GEOIDs from components
tract_geoid = construct_geoid(
    geography='tract',
    state='06',      # California
    county='037',    # Los Angeles
    tract='207400'   # Tract number
)
print(f"\nConstructed tract GEOID: {tract_geoid}")

# Parse a GEOID back to components
components = parse_geoid(tract_geoid, 'tract')
print(f"Parsed components: {components}")

# Normalize incomplete GEOIDs (adds leading zeros)
print(f"\nNormalize '6037' as county: {normalize_geoid('6037', 'county')}")
print(f"Normalize '6' as state: {normalize_geoid('6', 'state')}")

# Validate GEOIDs
print(f"\nIs '06037' a valid county GEOID? {validate_geoid('06037', 'county')}")
print(f"Is '6037' a valid county GEOID? {validate_geoid('6037', 'county')}")

### 6.3 GEOID Utilities

GEOIDs are standardized identifiers for Census geographies. These utilities help with normalization and joining.

In [ ]:
# Get California counties with income data - geometry + demographics in one GeoDataFrame
try:
    ca_income_gdf = get_census_data_with_geometry(
        year=2022,
        geography='county',
        state_fips='06',  # California
        variables=['B19013_001E']  # Median household income
    )
    
    print(f"GeoDataFrame with {len(ca_income_gdf)} counties")
    print(f"Columns: {list(ca_income_gdf.columns)}")
    
    # Create a choropleth map of median income
    fig, ax = plt.subplots(1, 1, figsize=(10, 12))
    ca_income_gdf.plot(
        column='B19013_001E',
        ax=ax,
        legend=True,
        legend_kwds={'label': 'Median Household Income ($)'},
        cmap='YlGnBu',
        edgecolor='black',
        linewidth=0.3
    )
    ax.set_title('California Counties - Median Household Income (2022 ACS)')
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"Note: Requires Census API. Error: {e}")

### 6.2 Combined Geometry + Demographics

The most powerful function: download boundaries AND demographics in one call.

In [ ]:
# Get population data for California counties (2022 ACS 5-year)
# Note: API key optional but recommended for higher rate limits
# Set CENSUS_API_KEY environment variable or pass api_key parameter

try:
    ca_pop = get_population(
        state='California',
        geography='county',
        year=2022
    )
    print(f"Retrieved population for {len(ca_pop)} California counties")
    print("\nTop 5 most populous counties:")
    print(ca_pop.nlargest(5, 'B01001_001E')[['NAME', 'B01001_001E']])
except Exception as e:
    print(f"Note: Census API may require an API key. Error: {e}")
    print("Get a free key at: https://api.census.gov/data/key_signup.html")

### 6.1 Quick Demographic Fetch (Convenience Functions)

These functions work without an API key for basic queries (rate-limited).

In [ ]:
# Import CensusAPIClient
import os
from siege_utilities.geo.census_api_client import (
    CensusAPIClient,
    get_demographics,
    get_population,
    get_income_data,
    get_census_data_with_geometry,
    VARIABLE_GROUPS
)

# Show available variable groups
print("Available predefined variable groups:")
for group_name, variables in VARIABLE_GROUPS.items():
    print(f"  {group_name}: {len(variables)} variables")

# 04: Spatial Data & Census Boundaries

This notebook demonstrates the spatial data capabilities of siege_utilities, including:

1. **Census Boundary Downloads** - Download TIGER/Line shapefiles
2. **State FIPS Normalization** - Convert between state names, abbreviations, and FIPS codes
3. **Geographic Level Discovery** - Find available boundary types for any year
4. **Working with GeoDataFrames** - Basic spatial operations

## Prerequisites

```bash
pip install -e /path/to/siege_utilities
```

In [ ]:
# Imports
import warnings
warnings.filterwarnings('ignore')

from siege_utilities.geo.spatial_data import (
    # Census data functions
    get_census_boundaries,
    download_data,
    get_available_years,
    get_year_directory_contents,
    discover_boundary_types,
    
    # State normalization
    normalize_state_identifier,
    get_state_by_abbreviation,
    get_state_by_name,
    
    # Global instances
    census_source
)

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

print("Imports successful!")

## 1. State FIPS Normalization

The `normalize_state_identifier()` function accepts any state format and returns the FIPS code.

In [ ]:
# Test various input formats
test_inputs = [
    '06',           # FIPS code
    'CA',           # Abbreviation
    'California',   # Full name
    'california',   # Lowercase
    'CALIFORNIA',   # Uppercase
    'TX',
    'New York',
    'ny',
]

print("State FIPS Normalization")
print("=" * 40)
for inp in test_inputs:
    fips = normalize_state_identifier(inp)
    print(f"{inp:15} -> {fips}")

In [ ]:
# Get full state info
ca_info = get_state_by_abbreviation('CA')
print("State info for CA:")
print(ca_info)

print()
tx_info = get_state_by_name('Texas')
print("State info for Texas:")
print(tx_info)

## 2. Discovering Available Census Data

Census TIGER/Line files are available for different years. Let's discover what's available.

In [ ]:
# Get available years
years = get_available_years()
print(f"Available Census years: {len(years)} years")
print(f"Range: {min(years)} - {max(years)}")
print(f"Recent years: {sorted(years)[-5:]}")

In [ ]:
# Discover boundary types for 2020
boundary_types = discover_boundary_types(2020)
print(f"Boundary types available for 2020: {len(boundary_types)}")
print("\nCommon types:")
common = ['STATE', 'COUNTY', 'TRACT', 'BG', 'PLACE', 'ZCTA', 'CD']
for bt in common:
    if bt in boundary_types:
        print(f"  - {bt}")

## 3. Downloading Census Boundaries

### 3.1 State Boundaries (National)

In [ ]:
# Download all US state boundaries
states_gdf = get_census_boundaries(
    year=2020,
    geographic_level='state'
)

print(f"Downloaded {len(states_gdf)} states/territories")
print(f"Columns: {list(states_gdf.columns)}")
print(f"CRS: {states_gdf.crs}")

In [ ]:
# Quick visualization of continental US
continental = states_gdf[
    ~states_gdf['statefp'].isin(['02', '15', '72', '78', '60', '66', '69'])
]

fig, ax = plt.subplots(1, 1, figsize=(12, 8))
continental.plot(ax=ax, edgecolor='black', facecolor='lightblue', linewidth=0.5)
ax.set_title('Continental United States')
ax.axis('off')
plt.tight_layout()
plt.show()

### 3.2 County Boundaries (State-specific)

In [ ]:
# Download county boundaries for California
ca_counties = get_census_boundaries(
    year=2020,
    geographic_level='county',
    state_fips='06'  # California
)

# Filter to just CA counties (national file includes all)
ca_counties = ca_counties[ca_counties['statefp'] == '06']

print(f"California has {len(ca_counties)} counties")
print("\nSample counties:")
print(ca_counties[['name', 'geoid', 'aland']].head(10))

In [ ]:
# Visualize California counties
fig, ax = plt.subplots(1, 1, figsize=(10, 12))
ca_counties.plot(ax=ax, edgecolor='black', facecolor='lightgreen', linewidth=0.5)
ax.set_title('California Counties')
ax.axis('off')
plt.tight_layout()
plt.show()

### 3.3 Census Tracts (County-specific)

In [ ]:
# Download census tracts for Los Angeles County
la_tracts = get_census_boundaries(
    year=2020,
    geographic_level='tract',
    state_fips='06'  # California
)

# Filter to LA County (FIPS 037)
la_tracts = la_tracts[la_tracts['countyfp'] == '037']

print(f"Los Angeles County has {len(la_tracts)} census tracts")
print(f"\nGeoid format: {la_tracts['geoid'].iloc[0]}")

In [ ]:
# Visualize LA County tracts
fig, ax = plt.subplots(1, 1, figsize=(12, 10))
la_tracts.plot(ax=ax, edgecolor='gray', facecolor='lightyellow', linewidth=0.1)
ax.set_title(f'Los Angeles County Census Tracts ({len(la_tracts)} tracts)')
ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Working with Geographic Data

### 4.1 Calculate Areas

In [ ]:
# Calculate area in square miles from ALAND (land area in sq meters)
ca_counties['area_sq_mi'] = ca_counties['aland'] / 2_589_988  # sq meters to sq miles

print("Largest California counties by area:")
print(ca_counties.nlargest(5, 'area_sq_mi')[['name', 'area_sq_mi']])

print("\nSmallest California counties by area:")
print(ca_counties.nsmallest(5, 'area_sq_mi')[['name', 'area_sq_mi']])

### 4.2 Spatial Joins

In [ ]:
# Create sample point data (cities in CA)
from shapely.geometry import Point

cities = gpd.GeoDataFrame({
    'city': ['Los Angeles', 'San Francisco', 'San Diego', 'Sacramento', 'Fresno'],
    'geometry': [
        Point(-118.2437, 34.0522),
        Point(-122.4194, 37.7749),
        Point(-117.1611, 32.7157),
        Point(-121.4944, 38.5816),
        Point(-119.7871, 36.7378)
    ]
}, crs='EPSG:4326')

# Spatial join to find which county each city is in
cities_with_county = gpd.sjoin(cities, ca_counties[['name', 'geometry']], how='left', predicate='within')
cities_with_county = cities_with_county.rename(columns={'name': 'county'})

print("Cities with their counties:")
print(cities_with_county[['city', 'county']])

## 5. Alternative Download Function

The `download_data()` function provides a simpler interface.

In [ ]:
# Using download_data() - equivalent functionality
tx_counties = download_data(
    year=2020,
    geographic_level='county',
    state_fips='48'  # Texas
)

tx_counties = tx_counties[tx_counties['statefp'] == '48']
print(f"Texas has {len(tx_counties)} counties")

## Summary (Part 1: Boundaries)

This section demonstrated:

| Function | Purpose |
|----------|----------|
| `normalize_state_identifier()` | Convert any state format to FIPS |
| `get_available_years()` | List available Census years |
| `discover_boundary_types()` | Find boundary types for a year |
| `get_census_boundaries()` | Download TIGER/Line shapefiles |
| `download_data()` | Simplified download interface |

**Key Parameters:**
- `year`: Census year (1992-present)
- `geographic_level`: state, county, tract, bg (block group), place, zcta, cd
- `state_fips`: Required for tract, bg, block levels

---

## Part 2: Census Demographics (CensusAPIClient)

The `CensusAPIClient` fetches demographic data from the Census Bureau API (ACS, Decennial).
It complements boundary downloads by providing actual demographic values.